<a href="https://colab.research.google.com/github/kasettisiva/R_training_summer_workshop_2026/blob/main/Introduction_to_R_AI_Era_ClaudeGenerated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

#
# **LONI Scientific Computing Bootcamp 2026**

## **Introduction to R**

*A hands-on workshop for data analysis, visualization, and statistical computing.*

---

**Siva Prasad Kasetti**

**LSU/LONI HPC User Services**  
**skasetti1@lsu.edu**  
**08 June 2026**

</div>

# **Notebook 4: Visualization, Statistics & Useful Functions**

This notebook covers data visualization with `ggplot2`, statistical analysis (correlation, regression, t-tests), useful R functions (apply family, plyr, user-defined functions), package management, and parallel processing.

**Sections:** ggplot2 · Statistics · Apply Family · plyr · User-defined Functions · Package Management · Parallel Processing

---

# **5 · Visualization with ggplot2**

ggplot2 uses a **grammar of graphics**: every plot is built from layers.

```
ggplot(data, aes(x = ..., y = ..., color = ...))  # canvas + mapping
  + geom_*()                                        # geometric layer
  + labs()                                          # labels
  + theme_*()                                       # styling
```

Once you know this grammar, you can read any ggplot2 code AI generates.

In [ ]:
# Scatter plot: weight vs fuel efficiency, colored by cylinders
ggplot(mtcars, aes(x = wt, y = mpg, color = factor(cyl))) +
  geom_point(size = 3, alpha = 0.8) +
  geom_smooth(method = "lm", se = FALSE) +  # regression line per group
  labs(
    title = "Fuel Efficiency vs Weight",
    subtitle = "By number of cylinders",
    x = "Weight (1000 lbs)",
    y = "Miles per Gallon",
    color = "Cylinders"
  ) +
  theme_minimal()

In [ ]:
# Box plot: distribution of mpg by cylinder count
ggplot(mtcars, aes(x = factor(cyl), y = mpg, fill = factor(cyl))) +
  geom_boxplot(alpha = 0.7, outlier.shape = 21) +
  labs(title = "MPG Distribution by Cylinder Count",
       x = "Cylinders", y = "Miles per Gallon") +
  theme_bw() +
  theme(legend.position = "none")

In [ ]:
# Histogram with density overlay
ggplot(mtcars, aes(x = mpg)) +
  geom_histogram(aes(y = after_stat(density)), bins = 12,
                 fill = "#4E79A7", color = "white", alpha = 0.8) +
  geom_density(color = "#E15759", linewidth = 1.2) +
  labs(title = "Distribution of Fuel Efficiency",
       x = "Miles per Gallon", y = "Density") +
  theme_minimal()

> **AI + ggplot2 workflow:**
> Describe your desired plot to AI in plain English.
> Paste the generated code here and run it.
> If it's wrong, copy the error message back to AI.
> Iterate until correct — then make sure you understand every line.

### **Quick exercise**

Ask Claude or ChatGPT: *"Write ggplot2 R code for a bar chart showing average horsepower
per cylinder count in mtcars, with a clean minimal theme."*

Paste the code below and verify it runs correctly.

In [ ]:
# Paste AI-generated ggplot2 code here and run it

---

# **6 · Statistics in R**

AI can run statistics for you. But you need to know *which test* to ask for
and whether the *result makes sense*.

## **6.0 Basics**

Getting to know your data is the first step in any analysis. R has a handful
of built-in functions that give you an instant snapshot — the smallest and
largest values, the average, how spread out the data is, and more. Think of
these as your quick sanity check before doing anything else.

### **Simple Statistic Functions**

* `min()`: Minimum value
* `max()`: Maximum value
* `which.min()`: Location of minimum value
* `which.max()`: Location of maximum value
* `sum()`: Sum of the elements of a vector
* `mean()`: Mean of the elements of a vector
* `sd()`: Standard deviation of the elements of a vector
* `quantile()`: Show quantiles of a vector
* `summary()`: Display descriptive statistics

In [ ]:
mean(mtcars$mpg)

In [ ]:
# The summary function will report a few statistics for each varialbe in the data frame.
summary(mtcars)

The `table()` function tabulates factors or find the frequency of an object.

For instance, in the `mtcars` data frame, we can get the frequency table by the combination of the numbers of cylinders and gears:

In [ ]:
table(mtcars[,c("cyl","gear")])

### **Distributions and Random Variables**

For each statistic distribution below, R provides four functions: density (d), cumulative density (p), quantile (q), and random generation (r).

Distrituion | Name in R

* Uniform | ```unif```
* Binomial | ```binom```
* Poisson | ```pois```
* Geometric | ```geom```
* Gamma | ```gamma```
* Normal | ```norm```
* Log Normal | ```lnorm```
* Exponential | ```exp```
* Student's t | ```t```

The function name is of the form `[d|p|q|r]<name of
distribution>`. For example,  `qbinom()` gives the quantile of a binomial distribution.

To generate a random sample of 10 from the standard normal distribution:

In [ ]:
# Each time it is run, the sample would be different.
rnorm(10,mean=0,sd=1)

[1]  0.70409930 -0.63609074  0.75451184 -0.42709339 -0.54699568 -1.58502271
 [7] -1.65112094 -1.84613813  0.01587029 -1.14035568

The p-value for 1.96 and its inverse function (standard normal distribution):

In [ ]:
pnorm(1.96)
qnorm(pnorm(1.96))

[1] 0.9750021

[1] 1.96

When genrating random samples, setting the seed to the same value will generate the same sample.

In [ ]:
# A "true" random sample (varies every time)
rnorm(10,mean=0,sd=1)

In [ ]:
# A sample with seed "15" (mainly for debugging purpose)
set.seed(15)
rnorm(10,mean=0,sd=1)
set.seed(15)
rnorm(10,mean=0,sd=1)

## **6.1 Correlation**

Correlation tells us how strongly two numeric variables are related.

For example, you can ask the question:
> Do heavier cars have lower fuel efficiency?

In [ ]:
# How strongly are weight and fuel efficiency correlated?
cor(mtcars$wt, mtcars$mpg)
# r = -0.87 → strong negative correlation (heavier cars get worse mileage)

In [ ]:
plot(mtcars$wt, mtcars$mpg,
     xlab = "Car Weight",
     ylab = "Miles Per Gallon",
     main = "Weight vs MPG")

## **6.2 Linear Regression**

Linear regression is used when we want to predict one numeric variable using another variable.

For example:

> Can we predict a car’s MPG using its weight?

In [ ]:
# Simple linear regression: predict mpg from weight
model_lm <- lm(mpg ~ wt, data = mtcars)
summary(model_lm)

Multiple regression uses more than one predictor.

For example:

> Can we predict MPG using weight, horsepower, and number of cylinders?

In [ ]:
# Multiple regression: add horsepower and cylinders
model_multi <- lm(mpg ~ wt + hp + cyl, data = mtcars)
summary(model_multi)

# Key things to read in summary():
# - Estimate: slope for each predictor
# - Pr(>|t|): p-value (< 0.05 is conventionally "significant")
# - R-squared: % of variance explained (0.84 = 84% here)
# - Residual standard error: typical prediction error

## **6.3 t-test: comparing two groups**

A t-test compares the average value between two groups.

For example:

> Do 4-cylinder cars and 8-cylinder cars have different average MPG?

In [ ]:
# Do 4-cylinder and 8-cylinder cars differ significantly in MPG?
cars_4cyl <- mtcars[mtcars$cyl ==4, "mpg"]
cars_8cyl <- mtcars[mtcars$cyl ==8, "mpg"]

t.test(cars_4cyl, cars_8cyl)
# p < 0.05 → yes, the difference is statistically significant

> **Statistical responsibility in the AI era:**
> AI can generate a p-value. Only you can decide if the question being tested is sensible,
> whether the data meets the test's assumptions, and what the result actually means
> in your domain. Statistics requires judgment, not just computation.

# **7. Useful Functions**

## **7.1 The Apply family of functions**

The `apply()` function evaluate a function over
the margins of an array
* More concise than the ```for``` loops (not necessarily
faster)

Syntax:

`
apply(data,dimension,function,function parameters)
 `

For example, if we want to calculate the mean of each variable in **mtcars**:

In [ ]:
# Apply the mean() function to the columns (variables) of mtcars.
apply(mtcars,2,mean)

*Which* is (almost) equivalent to:

In [ ]:
for (i in 1:ncol(mtcars)) {
  print(mean(mtcars[,i]))
}

It can perform multiple calculations in one function call:

In [ ]:
# Find the 1st and 3rd quantile of each varialbe in mtcars.
apply(mtcars, 2, quantile, probs = c(0.25, 0.5, 0.75))

Other member of the `apply()` family include:
* `lapply` - Loop over a list (data frame) and evaluate a function on each element
* `sapply` - Same as `lapply` but simplifies the result to array
* `tapply` - Apply a function over subsets of a vector
* `mapply` - Multivariate version of `sapply`

## **7.2 The ```plyr``` package**

Suppose that, with the **mtcars** data frame, we want to know the average mileage-per-gallon for cars with 4, 6 and 8 cylinders. How do we do that?

We will need to
* **split** the data into subsets according to the value of the "cyl" column (one for cyl==4, one for cyl==6 and one for cyl==8)
* **apply** the mean function to the "mpg" column
* **combine** the results from each subset

In [ ]:
# Subset where cyl==4
mean(mtcars[mtcars$cyl==4,"mpg"])
# Subset where cyl==6
mean(mtcars[mtcars$cyl==6,"mpg"])
# Subset where cyl==8
mean(mtcars[mtcars$cyl==8,"mpg"])

The "split-apply-combine" pattern is very common in data analysis, where you solve a complex problem by breaking it down into small pieces, doing something to each piece and then combining the results back together again.

The `plyr` packages provide a group of functions that implement this split-apply-combine pattern.

For example, the `ddply()` function takes a data frame, split it accorindg to the condition you supply, apply a function to each piece, then combine the result into a new data frame.

In [ ]:
library(plyr)
ddply(mtcars,"cyl", summarize, AverageMPG=mean(mpg))

The "split" step can be done according to more than one variable.

The command below will tell us the average MPG for each unique combination of "gear" and "cyl":

In [ ]:
# Find the average mpg for each unique combination of "gear" and "cyl".
ddply(mtcars,c("gear","cyl"), summarize, AverageMPG=mean(mpg))

## **7.3 User-defined functions**

* Users can define their own functions in R by using the ```function()``` directives.
* The return value is the last expression in the function body to be evaluated.
* Functions can be nested.
* Functions are R objects and can be passed as an argument to
other functions.

Syntax to define a function:


```
function_name <- function (arguments) {
  statements
}
```



In [ ]:
# Create a function pow(), which takes two arguments.
pow <- function(x, y) {
  result <- x^y
}

Then it can be called like any other function:

In [ ]:
# The result will be 4^2.
pow(4,2)

Functions can be used as an argument for other functions.

In [ ]:
# Define a new function, which takes a function as one of the arguments.
myfunc <- function(func,a,b) {
  result <- func(a,b) - 1
}

c <- myfunc(pow,4,2)
c

c <- myfunc(rep,1:2,5)
c

[1] 15

[1] 0 1 0 1 0 1 0 1 0 1

# **8 . Managing R Packages**

To load a R package so you can use the functions included in it, use the `library()` or `require()` function:

In [ ]:
library(lubridate)
require(devtools)

The main difference is that, if a package is not installed, `library()` will throw out an error message and the execution will stop, while `require()` throws out a warning and the execution continues.

In [ ]:
library(reshape)
print("End of code segment")

In [ ]:
require(reshape)
print("End of code segment")

If a package is not available, the `install.packages()` function can be used to install it.

In [ ]:
install.packages("reshape")
library(reshape)

Multiple packages can be installed with one call of the `install.packages()` function.

In [ ]:
require(datarium)
require(BiocManager)
install.packages(c("datarium","BiocManager"))
library(datarium)
library(BiocManager)

Note that double quotation is **NOT** needed when loading packages, but necessary when installing them.

Use the `remove.packages()` function to remove installed packages.

In [ ]:
remove.packages("datarium")
library(datarium)

The `update.packages()` function updates installed packages.

In [ ]:
update.packages("lubricate")

List **installed** packages.

In [ ]:
installed.packages()

List all **loaded** packages (and attached objects).

In [ ]:
search()

# **9 . Parallel Processing**

Modern computers are equipped with more than one CPU core and are capable of processing workloads in parallel, but base R is single‐threaded, i.e. not parallel.

In other words, regardless how many cores are available, R can only
use one of them.

There are two options to run R in parallel: **implicit** and **explicit**.


## **9.1 Implicit parallel processing**

Some functions in R can call parallel numerical libraries.

For instance, on the LONI QB-2 and QB-3 clusters most linear algebraic and related functions (e.g. linear regression, matrix decomposition, computing inverse and determinant of a matrix) leverage the multi‐threaded Intel MKL library.

In this case, no extra coding is needed to take advange of the multiple CPU cores - those functions will automatically use multiple cores when being called.

## **9.2 Explicit parallel processing**

If the implicit option is not available for what you'd like to do, some codes need to be written.

Here is an example of using the `%dopar%` directive in the `doParallel` package.

The workload is to generate 100 random samples, each with
1,000,000 observations from a standard normal distribution, then take a summary for each sample.

In [ ]:
iters <- 100

Below is the sequential version with a for loop. The `system.time()` function is used to measure how long it takes to process the workload.

In [ ]:
# This code segment shows us how long it takes to run on one core.
system.time(
for (i in 1:iters) {
  to.ls <- rnorm(1e6)
  to.ls <- summary(to.ls)
}
)

   user  system elapsed 
 13.596   0.820  14.529 

This is the parallel example with the `doParallel` package.

In [ ]:
# This code segment shows us how long it takes to run on all available cores.
library(doParallel)

# Obtain the number of cores available.
ncpu <- detectCores()
ncpu

system.time({
  cl <- makeCluster(ncpu)
  registerDoParallel(cl)
  ls<-foreach(icount(iters)) %dopar% {
    to.ls<-rnorm(1e6)
    to.ls<-summary(to.ls)
  }
  stopCluster(cl)
})

If you are interested to learn more, please visit the [Parallel Computing in R](http://hpc.loni.org/training/weekly-materials/2017-Fall/HPC_Parallel_R_Fall2017.pdf) tutorial from LONI HPC.